In [ ]:
from synthetic_train_pipeline import run_synthetic_training

import yaml

# Load your config
with open("configs/final_experiment_yamls/maskrcnn_resnet50_fpn_synth.yaml", "r") as f:
    cfg = yaml.safe_load(f)

# Train (progress printed + TensorBoard logging)
model, best_val, best_epoch, cfg_out, out_dir = run_synthetic_training(cfg)

loading annotations into memory...
Done (t=0.32s)
creating index...
index created!
loading annotations into memory...
Done (t=0.07s)
creating index...
index created!
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!

======== Starting Synthetic Training: maskrcnn_resnet50_fpn_synth ========


Epoch 1/2
LRs: [0.001, 0.0001]


Epoch: [0]:   0%|          | 0/525 [00:00<?, ?it/s]

Validating...
creating index...
index created!
Test:  [  0/300]  eta: 0:03:15  model_time: 0.4273 (0.4273)  evaluator_time: 0.0128 (0.0128)  time: 0.6522  data: 0.2088  max mem: 2400
Test:  [100/300]  eta: 0:00:39  model_time: 0.1360 (0.1698)  evaluator_time: 0.0145 (0.0195)  time: 0.1796  data: 0.0053  max mem: 2400
Test:  [200/300]  eta: 0:00:18  model_time: 0.1271 (0.1564)  evaluator_time: 0.0081 (0.0210)  time: 0.1686  data: 0.0053  max mem: 2400
Test:  [299/300]  eta: 0:00:00  model_time: 0.1206 (0.1562)  evaluator_time: 0.0076 (0.0198)  time: 0.1441  data: 0.0053  max mem: 2400
Test: Total time: 0:00:55 (0.1841 s / it)
Averaged stats: model_time: 0.1206 (0.1562)  evaluator_time: 0.0076 (0.0198)
Accumulating evaluation results...
DONE (t=0.07s).
Accumulating evaluation results...
DONE (t=0.05s).
IoU metric: bbox
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.848
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.970
 Average Pr

Epoch: [1]:   0%|          | 0/525 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78a115be03a0>
Traceback (most recent call last):
  File "/home/jotac431/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/home/jotac431/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
    if w.is_alive():
  File "/home/jotac431/miniconda3/envs/bubbleseg/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78a115be03a0>
Traceback (most recent call last):
  File "/home/jotac431/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/home/jotac431/.local/lib/python3.10/site-packages/torch/utils/data/datalo

Validating...
creating index...
index created!
Test:  [  0/300]  eta: 0:02:42  model_time: 0.3646 (0.3646)  evaluator_time: 0.0165 (0.0165)  time: 0.5415  data: 0.1573  max mem: 2400
Test:  [100/300]  eta: 0:00:28  model_time: 0.1075 (0.1185)  evaluator_time: 0.0111 (0.0157)  time: 0.1424  data: 0.0042  max mem: 2400
Test:  [200/300]  eta: 0:00:15  model_time: 0.1042 (0.1168)  evaluator_time: 0.0070 (0.0275)  time: 0.1390  data: 0.0044  max mem: 2400
Test:  [299/300]  eta: 0:00:00  model_time: 0.1129 (0.1178)  evaluator_time: 0.0071 (0.0235)  time: 0.1307  data: 0.0050  max mem: 2400
Test: Total time: 0:00:44 (0.1480 s / it)
Averaged stats: model_time: 0.1129 (0.1178)  evaluator_time: 0.0071 (0.0235)
Accumulating evaluation results...
DONE (t=0.05s).
Accumulating evaluation results...
DONE (t=0.06s).
IoU metric: bbox
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.848
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.980
 Average Pr

In [ ]:
from test_eval import build_test_dataloader, run_test_evaluation
from models import MODEL_REGISTRY
import torch
from datetime import datetime
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EVAL_SUITES = {
    "real_test": "../../../datasets/RealBubbles.coco-segmentation/test",
    "synth_test": "../../../BubANN_GAN.v1i.coco-segmentation/test"
}

eval_suite = "synth_test"

# Force all comparisons to use the same test set:
cfg["dataset"]["test"] = EVAL_SUITES[eval_suite]

# Build test loader
test_dl = build_test_dataloader(cfg)

#model_to_analyse = "maskrcnn_resnet50_fpn_v2_synth_20251120_1224"
model_to_analyse = out_dir

# Load model
model = MODEL_REGISTRY[cfg["model"]["type"]](cfg["model"]["num_classes"])
weights_path = os.path.join("outputs", model_to_analyse, "model_best.pth")
model.load_state_dict(torch.load(weights_path, map_location=device))
model.to(device)

# Evaluate
exp_name = cfg["experiment"]["name"]
out_dir = os.path.join("results", eval_suite, model_to_analyse)
results = run_test_evaluation(model, test_dl, device, cfg, out_dir, visualize=False)
results


loading annotations into memory...
Done (t=0.09s)
creating index...
index created!


TypeError: join() argument must be str, bytes, or os.PathLike object, not 'MaskRCNN'

In [ ]:
from plotting_module import run_all_plots_for_experiment

run_all_plots_for_experiment(out_dir)